# MSH chain on a 2D substrate: LDOS with tensor networks

**Goal.** The LDOS of the MSH chain on the substrate with the quantics MPO and MPS of `msh_substrate_tn.ipynb`, against the dense and
sparse reference of `msh_substrate_ldos_python.ipynb`, and its cost from 256² to 2048² sites.

**Four TN routes**, in the order they were tried:
1. **Zero-bias LDOS** from the TN zero mode $\psi_f$ (the zero-mode pipeline), read off the MPS along cuts and in a window: a row (or
   column) is made dense by fixing the y (or x) qubits, never the whole state.
2. **Whole-map KPM**: the Chebyshev operators $T_n(H)$ as MPOs, whose diagonals give the LDOS at every site and energy at once
   (TensorBinding's `KPM_Tn`, `get_ldos_diag_from_Tn`). Not viable: $\chi(T_n)$ reaches 256 by $n \approx 15$ (section 3).
3. **Per-site KPM**: the Chebyshev vectors $T_n(H)|\mathbf r, c\rangle$ as MPSs, giving the whole spectrum at one site (section 4).
4. **Subgap levels**: all states below the substrate gap by penalized DMRG, one level at a time on $(H - \sigma)^2 + \eta^2$, with their
   LDOS read off the MPSs (section 5). The benchmark point only (12 positive-energy levels); the leaking point's 53 would take days.

**Sizes and coordinates** as in the Python notebook: $N = N_xN_y$ lattice sites, $D = 4N$; cuts along the chain row ($i_x - x_0$) and across
it ($i_y - i_{y0}$). Results: `ldos_tn_zerobias.csv`, `ldos_tn_chi_Tn.csv`, `ldos_kpm_site.csv`, `ldos_tn_levels.csv` (committed) and
`data/ldos/` (cuts, moments, levels; not committed). All run flags are off (the runs took up to 2.3 h per size and were made outside the
notebook with the same functions, one process each); the figures comparing TN and sparse are in the Python notebook, section 6.

**Contents**
1. Setup
2. Zero-bias LDOS from the TN zero mode
3. Whole-map KPM: bond dimension of the Chebyshev MPOs
4. Per-site KPM with Chebyshev MPSs
5. Subgap levels by penalized DMRG
6. Summary

## 1. Setup

The LDOS helpers (`project_mps`, `row_weights`, `col_weights`, `subgap_levels`, `kpm_scale`, `site_kpm_moments`) are in `substrate_model.jl`.

In [ ]:
include("substrate_model.jl")
using NPZ, DelimitedFiles, Printf
BLAS.set_num_threads(1)
const DATA = "data/ldos"
mkpath(DATA)
const POINTS = Dict("bench" => (P = NamedTuple(), ℓ = 12), "leak" => (P = PC, ℓ = 64))
const H_WIN = 64
"CSV as (rows, column index)."
readcsv(f) = (A = readdlm(f, ','; header = true); (A[1], Dict(c => i for (i, c) in enumerate(vec(A[2])))))

readcsv (generic function with 1 method)

## 2. Zero-bias LDOS from the TN zero mode

`zerobias_tn(point, L)`: the zero-mode pipeline on $2^L\times2^L$ sites, then $w = 2\sum_c|\psi_{f,c}|^2$ on the chain row, across the chain at its
first site and its middle, and in a window of ±64 rows (each row by `row_weights`, the y qubits fixed). Written to
`data/ldos/zb_tn_<point>_<L>_<L>.npz`; times and memory to `ldos_tn_zerobias.csv`. The table compares the cuts with the sparse ones of the Python
notebook, section 2 (`zb_<point>_<L>_<L>.npz`).

In [ ]:
RUN_ZB = false
function zerobias_tn(pt, L)
    P, ℓ = POINTS[pt]; g = Substrate(L, L, ℓ); nx = Nx(g); x0, iy0 = first(chain_x(g)), chain_row(g)
    t_dmrg = @elapsed begin
        H0, sites = msh_substrate(g, S_z; P...)
        z = zero_mode_subspace(H0, sites; xq = xqubits(g)); r = fermion_from_subspace(H0, sites, z.ψs, g)
    end
    ψ = r.ψf / norm(r.ψf)
    cols = max(0, x0 - H_WIN):min(nx - 1, x0 + ℓ + H_WIN - 1); rows = max(0, iy0 - H_WIN):iy0 + H_WIN - 1
    t_cut = @elapsed begin
        row = 2 .* vec(sum(row_weights(ψ, sites, g, iy0, 0:nx-1); dims=1))
        cross_end = 2 .* vec(sum(col_weights(ψ, sites, g, x0, 0:nx-1); dims=1))
        win = 2 .* vcat([sum(row_weights(ψ, sites, g, iy, cols); dims=1) for iy in rows]...)
    end
    npzwrite(joinpath(DATA, "zb_tn_$(pt)_$(L)_$(L).npz"), Dict("row" => row, "cross_end" => cross_end, "win" => win, "x0" => [x0], "iy0" => [iy0], "ell" => [ℓ]))
    return (pt, nx, nx, t_dmrg, t_cut, Sys.maxrss() / 2^20, maxlinkdim(ψ))
end
if RUN_ZB
    open("ldos_tn_zerobias.csv", "w") do io
        println(io, "point,Nx,Ny,t_dmrg_s,t_cut_s,rss_MB,chi_psi")
        for pt in ("bench", "leak"), L in 8:11; println(io, join(zerobias_tn(pt, L), ",")); end
    end
end
Z, zc = readcsv("ldos_tn_zerobias.csv")
@printf("%6s %11s | %8s %6s %8s %5s | %22s\n", "point", "substrate", "DMRG", "cuts", "peak MB", "χ", "max |w_TN − w_sparse|")
for r in eachrow(Z)
    pt, n = r[zc["point"]], Int(r[zc["Nx"]]); L = round(Int, log2(n))
    t = npzread(joinpath(DATA, "zb_tn_$(pt)_$(L)_$(L).npz")); f = joinpath(DATA, "zb_$(pt)_$(L)_$(L).npz")
    d = isfile(f) ? (s = npzread(f); maximum(maximum(abs.(t[k] .- s[k])) for k in ("row", "cross_end", "win"))) : NaN
    @printf("%6s %5d×%-5d | %7.0fs %5.0fs %8.0f %5d | %22s\n", pt, n, n, r[zc["t_dmrg_s"]], r[zc["t_cut_s"]], r[zc["rss_MB"]], r[zc["chi_psi"]],
            isnan(d) ? "(no sparse vector)" : @sprintf("%.1e", d))
end

 point   substrate |     DMRG   cuts  peak MB     χ |  max |w_TN − w_sparse|
 bench   256×256   |     138s     9s     1426    43 |                1.8e-10


 bench   512×512   |     189s    10s     1558    43 |                6.4e-09
 bench  1024×1024  |     200s    12s     1612    43 |                3.5e-10
 bench  2048×2048  |     219s    14s     1548    43 |     (no sparse vector)
  leak   256×256   |    1370s     9s     1748   214 |                1.5e-10
  leak   512×512   |    1729s    12s     1877   211 |                7.6e-10
  leak  1024×1024  |    3685s    12s     2076   213 |                2.5e-10
  leak  2048×2048  |    2826s    14s     2163   211 |     (no sparse vector)


## 3. Whole-map KPM: bond dimension of the Chebyshev MPOs

TensorBinding's MPO KPM builds $T_{n+1} = 2\tilde H T_n - T_{n-1}$ ($\tilde H = H/s$, $s$ = `kpm_scale`) as MPOs and reads the LDOS at every site from
their diagonals. `chebyshev_mpo_chi` repeats the recursion with a cap of 256 and cutoff $10^{-10}$ and records $\chi(T_n)$ and the time per step.
At the benchmark point on $32\times32$ (512 moments for a resolution of 0.13), $\chi(T_n)$ reaches the cap by $n \approx 15$ and every later step
takes about 7 s; the full prototype (`KPM_Tn` plus `get_ldos_diag_from_Tn` at 61 energies) ran over 4 hours without finishing, against 48 s
for the exact KPM from dense eigenpairs. The operators $T_n(H)$ are not low-rank, so the whole-map route is not viable.

In [ ]:
RUN_CHI = false
function chebyshev_mpo_chi(pt, Lx, Ly, nmax; maxdim=256, cutoff=1e-10)
    P, ℓ = POINTS[pt]; g = Substrate(Lx, Ly, ℓ); H0, sites = msh_substrate(g, S_z; P...)
    Ht = H0 / kpm_scale(; (pt == "bench" ? NamedTuple() : (μ = PC.μ, Δ = PC.Δ, J = PC.J))...)
    Tm, T = MPO(sites, "Id"), Ht; out = Tuple{Int, Int, Float64}[]
    for n in 2:nmax
        dt = @elapsed (Tn = +(2 * apply(Ht, T; maxdim, cutoff), -1.0 * Tm; maxdim, cutoff))
        Tm, T = T, Tn; push!(out, (n, maxlinkdim(T), dt))
    end
    return out
end
if RUN_CHI
    open("ldos_tn_chi_Tn.csv", "w") do io
        println(io, "n,chi,step_s"); for o in chebyshev_mpo_chi("bench", 5, 5, 512); println(io, join(o, ",")); end
    end
end
X, xc = readcsv("ldos_tn_chi_Tn.csv")
for n in (2, 5, 10, 15, 50, 200, 400)
    i = findfirst(==(n), X[:, xc["n"]]); i === nothing && continue
    @printf("n = %3d  χ(T_n) = %3d  step %.2f s\n", n, X[i, xc["chi"]], X[i, xc["step_s"]])
end
@printf("steps 2–%d: %.0f s in total\n", X[end, xc["n"]], sum(X[:, xc["step_s"]]))

n =   2  χ(T_n) =  18  step 6.20 s
n =   5  χ(T_n) = 123  step 1.24 s


n =  10  χ(T_n) = 203  step 3.55 s
n =  15  χ(T_n) = 256  step 6.42 s
n =  50  χ(T_n) = 256  step 7.51 s
n = 200  χ(T_n) = 256  step 7.48 s
n = 400  χ(T_n) = 256  step 7.64 s
steps 2–418: 3063 s in total


## 4. Per-site KPM with Chebyshev MPSs

`site_kpm_moments`: the Chebyshev vectors of the four local states $|\mathbf r, c\rangle$ of the first chain site as MPSs (cap 512), 512 moments
with the doubling recursion. Unlike the operators, a single vector spreads by about one hop per step, so its bond dimension grows with the
number of moments (to about 175), not with the substrate. Compared with the sparse moments of the Python notebook, section 5. The 256²
run with cutoff $10^{-10}$ was overwritten by the $10^{-8}$ run, so its moments are compared at $10^{-8}$.

In [ ]:
RUN_KPM = false
if RUN_KPM
    for (L, cutoff) in ((8, 1e-10), (10, 1e-10))
        P, ℓ = POINTS["bench"]; g = Substrate(L, L, ℓ); H0, sites = msh_substrate(g, S_z; P...)
        x0, iy0 = first(chain_x(g)), chain_row(g)
        t = @elapsed ((μ, χ) = site_kpm_moments(H0, sites, g, x0, iy0, kpm_scale(); Ncheb=512, maxdim=512, cutoff))
        npzwrite(joinpath(DATA, "site_kpm_bench_$(L)_$(x0)_$(iy0)_n512_m512.npz"), Dict("mu" => μ, "chi" => [χ], "scale" => [kpm_scale()]))
        println("2^$L: $(round(Int, t)) s, χ $χ")
    end
end
K, kc = readcsv("ldos_kpm_site.csv")
@printf("%7s %7s %8s | %10s %6s %8s | %s\n", "size", "method", "cutoff", "s per site", "max χ", "peak MB", "LDOS error (relative)")
for r in eachrow(K)
    @printf("%6d² %7s %8s | %9.1fs %6s %8d | %s\n", r[kc["Nx"]], r[kc["method"]], string(r[kc["cutoff"]]), r[kc["secs_per_site"]],
            string(r[kc["max_chi"]]), r[kc["rss_MB"]], string(r[kc["ldos_rel_err"]]))
end
for (L, x0, iy0, lab) in ((8, 122, 128, "256², cutoff 1e-8"), (10, 506, 512, "1024², cutoff 1e-10"))
    f = joinpath(DATA, "site_kpm_bench_$(L)_$(x0)_$(iy0)_n512_m512")
    μt = npzread(f * ".npz")["mu"]; μs = npzread(f * "_sparse.npy")
    @printf("%s: max |μ_TN − μ_sparse| = %.1e\n", lab, maximum(abs.(μt .- μs)))
end

   size  method   cutoff | s per site  max χ  peak MB | LDOS error (relative)
   256²      TN  1.0e-10 |     304.0s    171     1617 | 


   256²  sparse          |      11.8s             281 | 
   256²      TN   1.0e-8 |     262.0s    151     1536 | 7.2e-5


  1024²      TN  1.0e-10 |     382.0s    175     1662 | 4.5e-6
  1024²  sparse          |     219.3s            3525 | 
256², cutoff 1e-8: max |μ_TN − μ_sparse| = 6.5e-04
1024², cutoff 1e-10: max |μ_TN − μ_sparse| = 4.4e-05


## 5. Subgap levels by penalized DMRG

`subgap_levels(H, sites, g; nlev = 12)`: the pair from the zero-mode pipeline, then each further level by DMRG on $K_\sigma = (H - \sigma)^2 + \eta^2$
with $\sigma$ just above the last level found and all found states and their partners penalized. With $K = H^2 + \eta^2$ alone, DMRG stalled at
the fifth level (an unconverged, nearly self-conjugate state with residual 0.76). A DMRG state mixes the $\pm E$ pair; diagonalizing $H$ on
span$\{\psi, C\psi\}$ separates them. `run_levels` saves the energies and the particle and hole weights of each level on the three cuts of the
Python notebook and in a ±64 window (`data/ldos/levels_tn_bench_<L>.npz`); the Python notebook, section 6, builds the LDOS from them.

In [ ]:
RUN_LEVELS = false
function run_levels(L; nlev=12)
    P, ℓ = POINTS["bench"]; g = Substrate(L, L, ℓ); nx = Nx(g); x0, iy0 = first(chain_x(g)), chain_row(g)
    H0, sites = msh_substrate(g, S_z; P...)
    lv = subgap_levels(H0, sites, g; nlev)
    cols = max(0, x0 - H_WIN):min(nx - 1, x0 + ℓ + H_WIN - 1); rows = max(0, iy0 - H_WIN):iy0 + H_WIN - 1
    ro = [(row_weights(φ, sites, g, iy0, cols), col_weights(φ, sites, g, x0, rows), col_weights(φ, sites, g, x0 + ℓ ÷ 2, rows),
           vcat([sum(row_weights(φ, sites, g, iy, cols); dims=1) for iy in rows]...)) for φ in lv.φs]
    npzwrite(joinpath(DATA, "levels_tn_bench_$L.npz"), Dict("E" => lv.E, "t" => lv.t, "chi" => Float64.(lv.chi),
        "row" => cat([o[1] for o in ro]...; dims=3), "cross_end" => cat([o[2] for o in ro]...; dims=3), "cross_mid" => cat([o[3] for o in ro]...; dims=3),
        "win" => cat([o[4] for o in ro]...; dims=3), "xcols" => collect(cols), "yrows" => collect(rows), "x0" => [x0], "iy0" => [iy0], "ell" => [ℓ]))
    return lv
end
RUN_LEVELS && foreach(run_levels, 8:11)

V, vc = readcsv("ldos_tn_levels.csv")
@printf("%6s | %8s %10s %8s | %22s\n", "size", "total", "max resid.", "χ", "max |E_TN − E_MUMPS|")
for n in (256, 512, 1024, 2048)
    sel = [r for r in eachrow(V) if r[vc["Nx"]] == n]
    tot = only(r[vc["secs"]] for r in sel if r[vc["level"]] == "total")
    lev = [r for r in sel if r[vc["level"]] != "total"]
    res = maximum(r[vc["residual"]] for r in lev if r[vc["residual"]] != ""); χs = [r[vc["chi"]] for r in lev]
    Et = sort(npzread(joinpath(DATA, "levels_tn_bench_$(round(Int, log2(n))).npz"))["E"])
    f = joinpath(DATA, "heat_sparse_bench_$(round(Int, log2(n))).npz")
    dE = isfile(f) ? (Es = npzread(f)["E"]; Es = sort(Es[Es .> 0]); @sprintf("%.1e", maximum(abs.(Et .- Es)))) : "(no MUMPS reference)"
    @printf("%5d² | %7.0fs %10.1e %3d–%-3d | %22s\n", n, tot, res, minimum(χs), maximum(χs), dE)
end
println("levels at 2048²: ", round.(sort(npzread(joinpath(DATA, "levels_tn_bench_11.npz"))["E"]); digits=6))

  size |    total max resid.        χ |   max |E_TN − E_MUMPS|
  256² |    1177s    7.4e-06  25–43  |                1.6e-09


  512² |    2348s    1.4e-05  25–43  |                6.0e-09
 1024² |    5786s    6.8e-06  25–43  |                1.3e-09
 2048² |    8343s    2.8e-06  25–43  |   (no MUMPS reference)
levels at 2048²: 

[0.000371, 0.552873, 0.564007, 0.595738, 0.694425, 0.903966, 1.217119, 1.596195, 1.998217, 2.379358, 2.696031, 2.90776]


## 6. Summary

- **Zero-bias LDOS.** The TN zero mode gives the in-gap LDOS along cuts and in a window around the chain in 9–14 s on top of the DMRG, at
  every size up to 2048², without a dense vector. It agrees with the sparse zero mode to $6\times10^{-9}$ (benchmark) and $8\times10^{-10}$ (leaking). Cost:
  DMRG 138–219 s and 1.4–1.6 GB at the benchmark point, 23–62 min and 1.7–2.2 GB at the leaking point; the sparse route is faster up to
  1024², but at 2048² only the TN is practical (MUMPS would need about 121 GB).
- **Whole-map KPM is not viable.** The Chebyshev operators $T_n(H)$ are not low-rank: $\chi(T_n)$ reaches 256 by $n \approx 15$ at 32×32.
- **Per-site KPM works but is slower than sparse up to 1024².** The Chebyshev vectors reach $\chi \approx 175$ at 512 moments, independent of the
  substrate: 304 s per site at 256² and 382 s at 1024² against 12 s and 219 s for sparse products ($\propto N$); the crossing is near 2048². LDOS
  error $5\times10^{-6}$ relative at cutoff $10^{-10}$; $10^{-8}$ saves only 14 %.
- **Subgap levels.** Penalized DMRG on $(H - \sigma)^2 + \eta^2$ finds all 12 positive-energy levels of the benchmark point at every size (residual
  $\approx 10^{-6}$, $\chi$ 25–34) in 20 min (256²) to 2.3 h (2048²) with 1.6–2.6 GB, agreeing with MUMPS to $10^{-9}$ in energy and $2\times10^{-5}$ in
  the LDOS cuts, and converged at 2048² against 1024². MUMPS is 5–14 times faster up to 1024²; at 2048² the TN gives the full subgap LDOS where
  MUMPS would need about 145 GB.
- **Recommendation.** Subgap LDOS by MUMPS up to 1024² and by TN levels beyond; zero-bias maps at 2048² by TN; spectra above the gap at a few
  sites by sparse KPM up to 1024² and TN per-site KPM from 2048²; no whole-map MPO KPM.